In [ ]:
import mysql.connector
import numpy as np
import pandas as pd
import zarr
import os

HEIGHT = 256
WIDTH = 384

In [ ]:
conn = mysql.connector.connect(
    user='root',
    password='',
    unix_socket='/tmp/mysql_database_dev.sock',
    database='heat_db'
)
cur = conn.cursor(dictionary=True)
cur.execute("SELECT DISTINCT img_file FROM pix01")
filenames = [row['img_file'] for row in cur.fetchall()]
print(f"{len(filenames)} ファイル")

9516 ファイル


In [ ]:
def fetch_img_file(filename):
    DN = np.zeros((HEIGHT, WIDTH))
    Modified_DN = np.zeros((HEIGHT, WIDTH))
    Mask = np.zeros((HEIGHT, WIDTH))
    for i in range(96):
        pix_num = f'pix{i + 1:02d}'
        cur.execute(f"SELECT * FROM {pix_num} WHERE img_file = %s", (filename,))
        for row in cur.fetchall():
            x, y = row['x'], row['y']
            DN[y, x] = np.nan if row['pixel'] is None else row['pixel']
            Modified_DN[y, x] = np.nan if row['pixel_modified'] is None else row['pixel_modified']
            Mask[y, x] = np.nan if row['mask'] is None else row['mask']
    return DN, Modified_DN, Mask

In [ ]:
# 3次元配列: (file_idx, y, x) = n_files × 256 × 384
base_dir = "/Volumes/Transcend/tileDB"
zarr_path = os.path.join(base_dir, "Haya2TIR.zarr")
os.makedirs(base_dir, exist_ok=True)

root = zarr.open(zarr_path, mode="w")
n_files = len(filenames)

# Zarr v3 では compressor を廃止。圧縮なしで作成（安定動作優先）
pixel = zarr.zeros((n_files, HEIGHT, WIDTH), chunks=(1, HEIGHT, WIDTH), dtype=np.int32, store=root.store, path="pixel")
pixel_modified = zarr.zeros((n_files, HEIGHT, WIDTH), chunks=(1, HEIGHT, WIDTH), dtype=np.int32, store=root.store, path="pixel_modified")
mask = zarr.zeros((n_files, HEIGHT, WIDTH), chunks=(1, HEIGHT, WIDTH), dtype=np.int32, store=root.store, path="mask")
print(f"Zarr 作成: {zarr_path}")

Zarr 作成: /Volumes/Transcend/tileDB/Haya2TIR.zarr


500/9516 完了
1000/9516 完了
1500/9516 完了
2000/9516 完了
2500/9516 完了
3000/9516 完了
3500/9516 完了
4000/9516 完了
4500/9516 完了
5000/9516 完了
5500/9516 完了
6000/9516 完了
6500/9516 完了
7000/9516 完了
7500/9516 完了
8000/9516 完了
8500/9516 完了
9000/9516 完了


KeyboardInterrupt: 

In [ ]:
for file_idx, filename in enumerate(filenames):
    DN, Modified_DN, Mask = fetch_img_file(filename)
    pixel[file_idx, :, :] = np.nan_to_num(DN, nan=0).astype(np.int32)
    pixel_modified[file_idx, :, :] = np.nan_to_num(Modified_DN, nan=0).astype(np.int32)
    mask[file_idx, :, :] = np.nan_to_num(Mask, nan=0).astype(np.int32)
    if (file_idx + 1) % 500 == 0:
        print(f"{file_idx + 1}/{n_files} 完了")
print("書き込み完了")

In [ ]:
pd.DataFrame({"file_idx": range(n_files), "filename": filenames}).to_csv(os.path.join(base_dir, "file_index_mapping.csv"), index=False)
print("file_index_mapping.csv 保存完了")